# PlantDoc + YOLOv8 - protocolo experimental do TCC

Este notebook executa o protocolo em fases e salva checkpoints no Google Drive. Antes de começar, selecione **Ambiente de execução > Alterar tipo de ambiente de execução > GPU**. T4 é suficiente para YOLOv8n; A100 reduz o tempo.

As fases podem ser executadas em sessões diferentes. Ao repetir uma célula, runs concluídos são detectados e não são treinados novamente.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
REPO = Path('/content/plantdoc-yolov8-tcc')
if not REPO.exists():
    !git clone https://github.com/SoulStorm0/plantdoc-yolov8-tcc.git {REPO}
else:
    !git -C {REPO} pull --ff-only
%cd /content/plantdoc-yolov8-tcc
!python -m pip install -q -e .

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError('GPU não detectada. Ative uma GPU no ambiente de execução do Colab.')
DEVICE = '0'
print('GPU:', torch.cuda.get_device_name(0))
print('PyTorch:', torch.__version__)

## 1. Preparação e auditoria do PlantDoc

O conversor lê a fonte oficial em Pascal VOC, cria nomes seguros, remove duas classes sem suporte estatístico mínimo e produz 27 classes com split determinístico 70/20/10. O conjunto de teste permanece isolado durante toda a seleção.

In [ ]:
SOURCE = Path('/content/plantdoc_official')
DATASET = Path('/content/plantdoc_yolo_27')
if not SOURCE.exists():
    !git clone --depth 1 --no-checkout https://github.com/pratikkayal/PlantDoc-Object-Detection-Dataset.git {SOURCE}
if not (DATASET / 'data.yaml').exists():
    !python scripts/prepare_official_plantdoc.py --repo {SOURCE} --output {DATASET} --min-class-instances 20
DATA_YAML = DATASET / 'data.yaml'
!python -m plantdoc_tcc audit --data {DATA_YAML} --split train --expected-classes 27
!python -m plantdoc_tcc audit --data {DATA_YAML} --split val --expected-classes 27
!python -m plantdoc_tcc audit --data {DATA_YAML} --split test --expected-classes 27

In [ ]:
RUN_ROOT = Path('/content/drive/MyDrive/TCC_PlantDoc/runs_staged')
RUN_ROOT.mkdir(parents=True, exist_ok=True)
PROTOCOL = REPO / 'configs/colab_protocol.json'
print('Checkpoints e resultados:', RUN_ROOT)
print('Protocolo:', PROTOCOL)

## 2. Fase A - comparação das funções de custo

Treina `baseline`, ponderação por classe e Focal Loss por 100 épocas com os mesmos hiperparâmetros. A melhor estratégia é escolhida por mAP@50:95 na validação.

In [ ]:
!python -m plantdoc_tcc staged --data {DATA_YAML} --config {PROTOCOL} --project {RUN_ROOT} --phase loss --device {DEVICE}

## 3. Fase B - busca de hiperparâmetros

Executa oito combinações planejadas por 100 épocas usando somente a loss vencedora. O espaço cobre batch 16/32, learning rate de 1e-3 a 1e-5, momentum 0,9/0,937 e weight decay de 1e-4 a 1e-3, com warmup e cosine annealing.

In [ ]:
!python -m plantdoc_tcc staged --data {DATA_YAML} --config {PROTOCOL} --project {RUN_ROOT} --phase search --device {DEVICE}

## 4. Fase C - promoção para 200 épocas

As três melhores combinações da validação são treinadas novamente por 200 épocas.

In [ ]:
!python -m plantdoc_tcc staged --data {DATA_YAML} --config {PROTOCOL} --project {RUN_ROOT} --phase promote200 --device {DEVICE}

## 5. Fase D - confirmação em 300 épocas

A melhor configuração de 200 épocas é confirmada em 300 épocas. Ainda é usada somente a validação.

In [ ]:
!python -m plantdoc_tcc staged --data {DATA_YAML} --config {PROTOCOL} --project {RUN_ROOT} --phase confirm300 --device {DEVICE}

## 6. Avaliação final no teste - execute uma única vez

Esta etapa calcula as métricas finais somente depois da seleção completa. O pipeline grava uma trava no resumo e bloqueia uma segunda avaliação, evitando ajuste indireto ao teste. Altere `CONFIRM_FINAL_TEST` para `True` apenas quando a Fase D estiver concluída.

In [ ]:
CONFIRM_FINAL_TEST = False
if not CONFIRM_FINAL_TEST:
    raise RuntimeError('Confirmação necessária: altere CONFIRM_FINAL_TEST para True.')
!python -m plantdoc_tcc final-test --data {DATA_YAML} --project {RUN_ROOT}

## 7. Resumo para o TCC

`protocol_runs.csv` contém todos os experimentos; `protocol_summary.json` registra a loss vencedora, o melhor modelo e a avaliação final; `final_test_metrics.json` contém métricas globais e por classe. Todos permanecem no Google Drive.

In [ ]:
import json
import pandas as pd
display(pd.read_csv(RUN_ROOT / 'protocol_runs.csv').sort_values('map50_95', ascending=False))
summary = json.loads((RUN_ROOT / 'protocol_summary.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2))